## Pre-processing the IO data
This code pre-processes the IO data, which is loaded into the code via csvs, into pickles and sparse matrices for later use. 

In [2]:
import pandas as pd
from scipy import sparse
import pickle

# Load the MRIO tables (i.e., GLORIA - downloadable from the ielab website)
file_path_T_2009 = '/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/20240111_120secMother_AllCountries_002_T-Results_2009_059_Markup001(full)(1).csv'
file_path_Y_2009 = '/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/20240111_120secMother_AllCountries_002_Y-Results_2009_059_Markup001(full)(1).csv'
file_path_V_2009 = '/Users/cmor7802/repos/climpyplots/boario/data/tmp/gloria59/20240111_120secMother_AllCountries_002_V-Results_2009_059_Markup001(full)(1).csv.csv'

# Load MRIO data using Pandas (include the year)
T_2009_df = pd.read_csv(file_path_T_2009, header=None)
Y_2009_df = pd.read_csv(file_path_Y_2009, header=None)
V_2009_df = pd.read_csv(file_path_V_2009, header=None)

# Convert DataFrame to sparse (sp) matrix (COO format) to pre-process for pickles
T_sp_2009 = sparse.coo_matrix(T_2009_df.values)
Y_sp_2009 = sparse.coo_matrix(Y_2009_df.values.squeeze()) 
V_sp_2009 = sparse.coo_matrix(V_2009_df.values)

# Pickle the sparse matrices for later use (i.e., reduce processing time)
with open('T_sp_2009.pkl', 'wb') as f:
    pickle.dump(T_sp_2009, f)
with open('Y_sp_2009.pkl', 'wb') as f:
    pickle.dump(Y_sp_2009, f)
with open('V_sp_2009.pkl', 'wb') as f:
    pickle.dump(V_sp_2009, f)

ImportError: Unable to import required dependencies:
numpy: Error importing numpy: you should not try to import numpy from
        its source directory; please exit the numpy source tree, and relaunch
        your python interpreter from there.

### Loading the pickled data

In [1]:
import pandas as pd
from scipy import sparse
import pickle

# Load the pickled sparse matrices
with open('T_sp_2022.pkl', 'rb') as f:
    T_sp_2022 = pickle.load(f)
with open('Y_sp_2022.pkl', 'rb') as f:
    Y_sp_2022 = pickle.load(f)
with open('V_sp_2022.pkl', 'rb') as f:
    V_sp_2022 = pickle.load(f)
with open('TQ_sp_2022.pkl', 'rb') as f:
    TQ_sp_2022 = pickle.load(f)


### Convert the pickled matrices to sparse format
This code loads in the sparse matrices and converts them to csr format (compressed sparse rows). This enables the further analysis and calculating X. 

In [2]:
# Load the regular pickled sparse matrix
with open('T_sp_2022.pkl', 'rb') as f:
    T_coo = pickle.load(f)
with open('Y_sp_2022.pkl', 'rb') as f:
    Y_coo = pickle.load(f)
with open('V_sp_2022.pkl', 'rb') as f:
    V_coo = pickle.load(f)
with open('TQ_sp_2022.pkl', 'rb') as f:
    TQ_coo = pickle.load(f)

# Convert the COO matrix to CSR format
T_csr_2022 = T_coo.tocsr()
Y_csr_2022 = Y_coo.tocsr()
V_csr_2022 = V_coo.tocsr()
TQ_csr_2022 = TQ_coo.tocsr()

# Save the CSR matrix back to a file for later use
with open('T_csr_2022.pkl', 'wb') as f:
    pickle.dump(T_csr_2022, f)
with open('Y_csr_2022.pkl', 'wb') as f:
    pickle.dump(Y_csr_2022, f)
with open('V_csr_2022.pkl', 'wb') as f:
    pickle.dump(V_csr_2022, f)
with open('TQ_csr_2022.pkl', 'wb') as f:
    pickle.dump(TQ_csr_2022, f)

## Slicing the T and Y matrices to ignore SUPPLY matrices
Both T and Y matrices require slicing to remove the supply matrices. 

For T, the use matrices (demand between sectors in one region) is: 
| R1 | R2 |
|---------|---------|
| 0   | supply   |
| use   | 0   |

Where the trade matrices are everywhere else. For disaster analysis, we need both use and trade matrices. 

For Y, we need to skip every 120 rows as they correlate to the supply matrices. Supply matrices have no final demand, so there are only 0s in these matrices and thus need to be removed. The structure is: 

| Y1 | Y2 |
|---------|---------|
| R1Y1   | R1Y2   |
| 0   | 0   |
| R2Y1   | R2Y2   |
| 0   | 0   |

Where each sector and regions Y, needs to be summed to get the total final demand. We can isolate the final demand sectors pending on the analysis. However, we only use total final demand. 

In [3]:
from scipy.sparse import csr_matrix
import numpy as np
import pickle

# Define block size and total dimensions
block_size = 120
total_rows, total_cols = T_csr_2022.shape

# Function to calculate indices to include based on the pattern
def calculate_indices(total_size, block_size, include_first):
    indices = []
    start = 0 if include_first else block_size
    for i in range(start, total_size, 2 * block_size):
        end = min(i + block_size, total_size)
        indices.extend(range(i, end))
    return indices

# Calculate row and column indices to include
row_indices = calculate_indices(total_rows, block_size, include_first=False)  # Skip first batch of rows
col_indices = calculate_indices(total_cols, block_size, include_first=True)  # Include first batch of columns

# Extract the rows and columns based on calculated indices
new_matrix = T_csr_2022[row_indices, :][:, col_indices]

# Optionally, convert to CSR format if needed for further operations
new_matrix_csr = new_matrix.tocsr()

# Print dimensions and non-zero elements of the new matrix for verification
print("New Matrix Dimensions:", new_matrix_csr.shape)
print("Non-Zero Elements:", new_matrix_csr.nnz)

# Optionally, save the new matrix to a file
with open('Tsliced_2022.pkl', 'wb') as f:
    pickle.dump(new_matrix_csr, f)

New Matrix Dimensions: (19680, 19680)
Non-Zero Elements: 381188643


In [4]:
with open('Tsliced_2022.pkl', 'rb') as f:
    Tsliced_2022 = pickle.load(f)
    
print("Dimensions of T:", T_csr_2022.shape)
print("Dimensions of sliced T:", Tsliced_2022.shape)

Dimensions of T: (39360, 39360)
Dimensions of sliced T: (19680, 19680)


In [6]:
from scipy.sparse import csr_matrix
import numpy as np
import pickle

# Define block size and total dimensions
block_size = 120  # Each block size of 120 rows
total_rows, total_cols = Y_csr_2022.shape

# Function to calculate indices to include based on the pattern
def calculate_indices(total_size, block_size, skip_first):
    indices = []
    start = block_size if skip_first else 0  # Skip the first block if 'skip_first' is True
    for i in range(start, total_size, 2 * block_size):
        end = min(i + block_size, total_size)
        indices.extend(range(i, end))
    return indices

# Calculate row indices to include (skip the first block of 120 rows)
row_indices = calculate_indices(total_rows, block_size, skip_first=True)

# Extract the rows based on calculated indices (all columns are included)
new_Ymatrix = Y_csr_2022[row_indices, :]

# Convert to CSR format if needed for further operations
new_Ymatrix_csr = new_Ymatrix.tocsr()

# Print dimensions and non-zero elements of the new matrix for verification
print("New Matrix Dimensions:", new_Ymatrix_csr.shape)
print("Non-Zero Elements:", new_Ymatrix_csr.nnz)

# Optionally, save the new matrix to a file
with open('Ysliced_2022.pkl', 'wb') as f:
    pickle.dump(new_Ymatrix_csr, f)

New Matrix Dimensions: (19680, 984)
Non-Zero Elements: 18949235


In [7]:
with open('Ysliced_2022.pkl', 'rb') as f:
    Ysliced_2022 = pickle.load(f)
    
print("Dimensions of Y:", Y_csr_2022.shape)
print("Dimensions of sliced Y:", Ysliced_2022.shape)

Dimensions of Y: (39360, 984)
Dimensions of sliced Y: (19680, 984)


In [13]:
# Using .data to access the non-zero elements of the sparse matrix
print("Statistics of non-zero elements in T:")
print("Min:", Tsliced_2022.data.min())
print("Max:", Tsliced_2022.data.max())
print("Mean:", Tsliced_2022.data.mean())

print("Statistics of non-zero elements in Y:")
print("Min:", Ysliced_2022.data.min())
print("Max:", Ysliced_2022.data.max())
print("Mean:", Ysliced_2022.data.mean())

Statistics of non-zero elements in T:
Min: 3.2979e-11
Max: 967630000.0
Mean: 269.54171496866735
Statistics of non-zero elements in Y:
Min: 1e-10
Max: 2900200000.0
Mean: 5508.943945339992


In [14]:
# this code converts the sparse matrices to dencse format and testes to see if T or Y are the issues in producing negative X
import numpy as np
import pickle

# Convert the sparse matrices to dense format
Tdsliced_2022 = Tsliced_2022.todense()
Ydsliced_2022 = Ysliced_2022.todense()

# Save the CSR matrix back to a file for later use
with open('Tdsliced_2022.pkl', 'wb') as f:
    pickle.dump(Tdsliced_2022, f)
with open('Ydsliced_2022.pkl', 'wb') as f:
    pickle.dump(Ydsliced_2022, f)
    
# Print some details to inspect
print("Shape of T:", Tdsliced_2022.shape)
print("Shape of Y:", Ydsliced_2022.shape)
print("First few rows of T:")
print(Tdsliced_2022[:5, :5])  # Adjust the slicing as needed
print("First few elements of Y:")
print(Ydsliced_2022[:5, :])  # Adjust the slicing as needed

# Optionally, check the statistics of the dense matrices
print("Statistics of T:")
print("Min:", np.min(Tdsliced_2022))
print("Max:", np.max(Tdsliced_2022))
print("Mean:", np.mean(Tdsliced_2022))

print("Statistics of Y:")
print("Min:", np.min(Ydsliced_2022))
print("Max:", np.max(Ydsliced_2022))
print("Mean:", np.mean(Ydsliced_2022))

Shape of T: (19680, 19680)
Shape of Y: (19680, 984)
First few rows of T:
[[4.9066e+01 1.7523e-06 2.4942e-06 2.3685e-06 1.8048e-06]
 [2.6926e-06 6.8804e-03 3.8568e-06 3.9965e-06 2.6347e-06]
 [2.8450e-06 2.9858e-06 1.0339e+03 5.1906e-06 3.0091e-06]
 [3.8283e-06 4.2509e-06 6.8151e-06 1.0122e+01 4.2361e-06]
 [2.6463e-06 2.5505e-06 3.8173e-06 3.9427e-06 1.1587e+01]]
First few elements of Y:
[[1.2568e+02 3.4954e-01 1.1346e+01 ... 5.7966e-07 2.8196e-07 5.8496e-04]
 [1.6461e-02 5.2176e-01 1.6936e+01 ... 3.9558e-03 3.2280e-05 8.7316e-04]
 [1.0650e+03 4.5065e-01 1.4628e+01 ... 6.1025e-07 2.9681e-07 7.5416e-04]
 [4.7062e+02 5.5877e-01 1.8137e+01 ... 9.4625e-07 4.6017e-07 9.3510e-04]
 [1.7713e+04 5.3654e-01 1.7416e+01 ... 8.3518e-07 4.0617e-07 8.9789e-04]]
Statistics of T:
Min: 0.0
Max: 967630000.0
Mean: 265.28686773125713
Statistics of Y:
Min: 0.0
Max: 2900200000.0
Mean: 5390.633955383425


## Slicing the VA matrix
GLORIA's VA matrix is in the format of $6*164$ x $240*164$. Each regions VA matrix has 6 rows and 120 columns. This data is on the diagonal of the matrix, separated by 120 columns. The row index increases incrementally by 6, and the columns skip 120 columns incrementally. 

In [58]:
import numpy as np
import pickle
from scipy.sparse import csr_matrix

# Load the existing sparse matrix from a pickle file
with open('V_csr_2022.pkl', 'rb') as f:
    V_csr_2022 = pickle.load(f)

# Define the total number of regions
num_regions = 164

def reshape_va_matrix(va_matrix, num_regions, row_block_size=6, col_block_size=120):
    """
    Extracts blocks from a sparse matrix based on a shifting pattern and reshapes them
    into a single matrix with 6 rows and 19680 columns (120*164).

    Parameters:
    - va_matrix: Sparse matrix in CSR format containing the VA data.
    - num_regions: Integer, number of regions (blocks).
    - row_block_size: Integer, the number of rows in each block.
    - col_block_size: Integer, the number of columns in each block.

    Returns:
    - numpy.ndarray: Reshaped array of all VA data.
    """
    # Calculate total columns
    total_columns = col_block_size * num_regions
    reshaped_matrix = np.zeros((row_block_size, total_columns))

    for i in range(num_regions):
        # Compute start row for each block
        start_row = i * row_block_size
        # Compute start and end column for each block
        start_col = i * (col_block_size + 120)  # Starting column index increases by block size + 120 for each region
        end_col = start_col + col_block_size

        # Extract the block and convert to a dense array
        block = va_matrix[start_row:start_row + row_block_size, start_col:end_col].toarray()

        # Place the block in the corresponding columns of the reshaped matrix
        col_offset = i * col_block_size
        reshaped_matrix[:, col_offset:col_offset + col_block_size] = block

    return reshaped_matrix

# Reshape the VA data from the matrix
reshaped_va_matrix = reshape_va_matrix(V_csr_2022, num_regions)
print("Reshaped VA Matrix Shape:", reshaped_va_matrix.shape)

# Optionally convert the reshaped array to a CSR matrix and save it
Vsliced_2022 = csr_matrix(reshaped_va_matrix)
with open('Vsliced_2022.pkl', 'wb') as f:
    pickle.dump(Vsliced_2022, f)

print("Reshaped VA matrix saved successfully in CSR format.")

Reshaped VA Matrix Shape: (6, 19680)
Reshaped VA matrix saved successfully in CSR format.


In [59]:
# Print the statistics of V to inspect
print("Statistics of VA:")
print("Min:", np.min(Vsliced_2022))
print("Max:", np.max(Vsliced_2022))
print("Mean:", np.mean(Vsliced_2022))

Statistics of VA:
Min: -1913600.0
Max: 1696400000.0
Mean: 883926.3458784332


In [60]:
## Code saves the file as a csv for inspection
import pandas as pd
import pickle

# Load the Vsliced_2022 matrix from the pickle file
with open('Vsliced_2022.pkl', 'rb') as f:
    Vsliced_2022 = pickle.load(f)

# Convert the CSR matrix to a dense format
dense_matrix = Vsliced_2022.toarray()

# Convert the dense matrix to a DataFrame
df = pd.DataFrame(dense_matrix)

# Optionally, label the columns and rows if necessary
column_names = [f"Sector {i+1}" for i in range(dense_matrix.shape[1])]
df.columns = column_names
row_names = [f"Category {i+1}" for i in range(dense_matrix.shape[0])]
df.index = row_names

# Save the DataFrame to a CSV file
csv_file_path = 'Vsliced_2022.csv'
df.to_csv(csv_file_path)

print(f"Matrix saved to CSV file at {csv_file_path}")

Matrix saved to CSV file at Vsliced_2022.csv


## Slicing TQ
This is the satellite accounts for the intermediate demand matrix. 

Row 368 = Female employment in k ppl
Row 369 = Male employment in k ppl

In [6]:
import numpy as np
import pickle
from scipy.sparse import csr_matrix

# Assume TQ_csr_2022 is loaded and ready
# For example, to load from a pickle file:
# with open('TQ_csr_2022.pkl', 'rb') as f:
#     TQ_csr_2022 = pickle.load(f)

# Define block size and total dimensions
block_size = 120
total_rows, total_cols = TQ_csr_2022.shape

def calculate_indices(total_size, block_size, include_first=True):
    """
    Calculates indices to include every second block of a specified size in a total dimension.

    Parameters:
    - total_size: Total number of columns or rows in the matrix.
    - block_size: Size of each block.
    - include_first: Boolean to determine whether to start with the first block or skip it.

    Returns:
    - list of indices to include.
    """
    indices = []
    start = 0 if include_first else block_size
    for i in range(start, total_size, 2 * block_size):
        end = min(i + block_size, total_size)
        indices.extend(range(i, end))
    return indices

# Calculate column indices to include
col_indices = calculate_indices(total_cols, block_size)

# Extract the rows and columns based on calculated indices
new_matrix = TQ_csr_2022[:, col_indices]

# Optionally, convert to CSR format if needed for further operations
TQsliced_2022 = new_matrix.tocsr()

# Print dimensions and non-zero elements of the new matrix for verification
print("New Matrix Dimensions:", TQsliced_2022.shape)
print("Non-Zero Elements:", TQsliced_2022.nnz)

# Optionally, save the new matrix to a file
with open('/Users/cmor7802/repos/disasterassessment/pkls/TQ/TQsliced_2022.pkl', 'wb') as f:
    pickle.dump(TQsliced_2022, f)

New Matrix Dimensions: (5982, 19680)
Non-Zero Elements: 2408371


In [7]:
print(TQ_csr_2022.shape)
print(TQsliced_2022.shape)

(5982, 39360)
(5982, 19680)


In [12]:
from scipy.sparse import load_npz

# Specify the rows and column of interest
rows_of_interest = [358, 359]
column_of_interest = 15720

# Extract the values at these indices
values = TQsliced_2022[rows_of_interest, column_of_interest].toarray()

# Convert to DataFrame for better visualization
import pandas as pd
values_df = pd.DataFrame(values, index=['Row 358', 'Row 359'], columns=['Column 15720'])
print(values_df)


         Column 15720
Row 358           0.0
Row 359           0.0


## Calculating total output (X)
This code calculates the total output for the pre-disaster economy. 

In [15]:
import numpy as np
import scipy.sparse as sp
import pickle

# Load Tsliced_2022 and Ysliced_2022
with open('Tsliced_2022.pkl', 'rb') as f:
    T = pickle.load(f)  # This should be a sparse matrix
with open('Ysliced_2022.pkl', 'rb') as f:
    C_dense = pickle.load(f)  # This should be a dense matrix, sum it to create a vector

# Ensure C is properly formatted as a column vector
if C_dense.ndim == 2:
    C = np.sum(C_dense, axis=1)  # Sum across columns to collapse it into a single vector
    C = C.reshape(-1, 1)  # Ensure it's a column vector
else:
    raise ValueError("Check the shape of Ysliced_2022; it should be a 2D matrix.")

# Calculate total output X
if sp.issparse(T):
    intermediate_demand = np.array(T.sum(axis=0)).flatten()  # Sum along columns for intermediate demand
else:
    intermediate_demand = np.sum(T, axis=0)  # Sum along columns for dense T

# Ensure intermediate demand is a column vector
intermediate_demand = intermediate_demand.reshape(-1, 1)

# Calculate total output X as the sum of intermediate demand and final demand C
X = intermediate_demand + C

# Check and print the shape of X
print("Dimensions of X:", X.shape)

# If the shape of X is not (19680, 1), adjust accordingly
if X.shape != (19680, 1):
    raise ValueError(f"Unexpected shape of X: {X.shape}, expected (19680, 1).")

# Save X for future use
with open('X_vector.pkl', 'wb') as f:
    pickle.dump(X, f)

Dimensions of X: (19680, 1)


In [16]:
print("Statistics of X:")
print("Min:", np.min(X))
print("Max:", np.max(X))
print("Max:", X.shape)

Statistics of X:
Min: 0.0
Max: 4696490953.14015
Max: (19680, 1)


## A (technical coefficients matrix) calculation
Calculating A requires the X and T matrices, where $A = T(X^-1)$. 

In [1]:
import numpy as np
import scipy.sparse as sp
import pickle

# Load the total output vector x
with open('X_vector.pkl', 'rb') as f:
    x = pickle.load(f)

# Ensure x is a column vector if it's not already
if x.ndim == 1:
    x = x.reshape(-1, 1)  # Reshape x to be a column vector (19680, 1)
elif x.shape[1] != 1:
    x = x.reshape(-1, 1)  # Ensure it's reshaped correctly if it's not a 1D array

# Invert x safely, avoiding division by zero
x_inv = np.where(x != 0, 1.0 / x, 0)
x_inv[np.isinf(x_inv)] = 0  # Replace infinities if any

# Flatten x_inv to ensure it's a 1D array suitable for diags
x_inv = x_inv.flatten()

# Create a diagonal matrix from the inverted x
X_diag = sp.diags(x_inv, offsets=0, format='csr')

# Load the intermediate demand matrix Z (or T)
with open('Tsliced_2022.pkl', 'rb') as f:
    Z = pickle.load(f)

# Compute A by multiplying Z with the inverted diagonal matrix of x
A = Z.dot(X_diag)

# Save the computed A matrix
with open('A_matrix.pkl', 'wb') as f:
    pickle.dump(A, f)

# Print some statistics to verify
print("Shape of x:", x.shape)
print("Shape of X_diag:", X_diag.shape)
print("Shape of A:", A.shape)
print("Maximum value in A:", A.data.max())
print("Minimum value in A:", A.data.min())

/var/folders/s4/pb34svzd4vv8yhd414jr_hkh0000gq/T/ipykernel_3307/3240367456.py:16: RuntimeWarning: divide by zero encountered in divide
  x_inv = np.where(x != 0, 1.0 / x, 0)


Shape of x: (19680, 1)
Shape of X_diag: (19680, 19680)
Shape of A: (19680, 19680)
Maximum value in A: 0.9965429981528874
Minimum value in A: 2.809948946159773e-20


## Slicing the matrix to extract certain regions T data
This code slices the sparse matrix to isolate certain regions T data to produce the analysis. 

To calculate the regions, we have to consider Python indexing which starts at 0. This means that each region is their correlating region number -1. 

Calculate the Start Index: Multiply this adjusted position by the number of sectors (rows/columns) per region. So if Libya is the 92nd region, then it really is the 91st. 
- Start Index=(92−1)×120=91×120=10920
- End Index=Start Index+120−1=10920+120−1=11039

First, we slice using csr process on the sparse matrix we loaded in. Then we convert it back to a dense format. We then compare the results by slicing data in the pandas dataframe. This test helps us identify if we are slicing the right data. 

In [ ]:
import pickle
from scipy.sparse import csr_matrix
import numpy as np

regions = {
    'Rest of Americas': (0, 119),
    'Libya': (10920, 11039),
    'Ukraine': (18480, 18600),
    'United States': (18720, 18840)
    }

for region_name, (start_index, end_index) in regions.items():
    # Slice the matrix
    region_matrix_sparse = T_csr_2022[start_index:end_index+1, start_index:end_index+1]
    
    # Convert to dense array
    region_matrix_dense = region_matrix_sparse.toarray()
    
    # # Save the dense matrix using pickle
    # with open(f'{region_name}_dense_matrix.pkl', 'wb') as f:
    #     pickle.dump(region_matrix_dense, f)


Here we want to aggregate sectoral data from multiple regions into one region. For example, we want to take all the data for each individual small island developing state (SIDS) and make it into one new matrix called SIDS. 

In [ ]:
import pickle
from scipy.sparse import csr_matrix
import numpy as np

sids_regions = {
    'Costa Rica': (4680, 4800),
    'Mauritania': (12720, 12840),
    'Panama': (14400, 14520),
    'Philippines': (14640, 14760),
    'Papua New Guinea': (14760, 14880),
    'Singapore': (16200, 16320)
}

# Assuming each country contributes equally to the number of sectors
num_sectors = 120
sids = csr_matrix((1, num_sectors))

with open('T_csr_2022.pkl', 'rb') as f:
    T_csr_2022 = pickle.load(f)

for country, (start_index, end_index) in sids_regions.items():
    # Extract the block from the original matrix
    country_data = T_csr_2022[start_index:end_index+1, :]

    # Sum the rows to condense each country's data into a single row
    country_aggregated = country_data.sum(axis=0)

    # Since we are extracting the whole row, make sure to limit columns to the first 120 if the matrix is larger
    if country_aggregated.shape[1] > num_sectors:
        country_aggregated = country_aggregated[:, :num_sectors]

    # Add this country's aggregated data to the "SIDS" row
    sids += country_aggregated

# with open('sids_T_2022.pkl', 'wb') as f:
#     pickle.dump(sids, f)

print(sids.shape)  # Should print (1, 120)

(1, 120)


In [ ]:
import pickle
import numpy as np

# Load the original sparse matrix
with open('T_csr_2022.pkl', 'rb') as f:
    T_csr = pickle.load(f)

# Load the sliced sparse matrix for Libya
with open('Libya_dense_matrix.pkl', 'rb') as f:
    Libya_matrix = pickle.load(f)

# Get Libya indices from the dictionary
libya_indices = regions['Libya']
libya_start_index, libya_end_index = libya_indices

# Extract the slice for Libya from the original matrix
libya_slice = T_csr[libya_start_index:libya_end_index+1, libya_start_index:libya_end_index+1].toarray()

# # Load the previously saved dense matrix for Libya
# with open('Libya_dense_matrix.pkl', 'rb') as f:
#     libya_dense_matrix = pickle.load(f)

# Compare the two matrices
comparison_result = np.array_equal(libya_slice, libya_dense_matrix)
print("Comparison result:", comparison_result)


Comparison result: True


In [ ]:
from scipy.sparse import csr_matrix
import numpy as np
import pickle

# Assuming T is loaded
num_regions = 164  # Total number of regions
rows_per_region = 240  # Total number of sectors per region (120 - supply, 120 - use)
use_rows_per_region = 120  # Number of sectors for the use matrix
cols_per_region = 240  # Total number of columns per region (120*2 per region)
use_cols_per_region = 120  # Number of columns for the use matrix including trade see https://www.dropbox.com/sh/o4fxq94n7grvdbk/AAAvqYty44k-0VoE87J1OQcba/latest_release?e=1&preview=GLORIA_ReleaseNotes_059.pdf&subfolder_nav_tracking=1&dl=0

total_rows, total_cols = T_csr_2022.shape  # Get the total dimensions of T
use_tables = {}

for i in range(num_regions):
    start_row = i * rows_per_region + use_rows_per_region  # Starting from the second block of 120 rows
    end_row = start_row + use_rows_per_region
    start_col = i * cols_per_region  # Starting from the first column
    end_col = start_col + use_cols_per_region

    # Ensure that the indices do not exceed the matrix dimensions
    if end_row > total_rows:
        end_row = total_rows
    if end_col > total_cols:
        end_col = total_cols

    # Extract the use table for each region
    use_table = T_csr_2022[start_row:end_row, start_col:end_col].tocoo().tocsr()
    use_tables[f"Region {i+1}"] = use_table

    # # Optionally, save each use table as a pickle file for later use
    # with open(f'use_table_region_{i+1}.pkl', 'wb') as f:
    #     pickle.dump(use_table, f)

# Print the dimensions of each region's use table
for region, matrix in use_tables.items():
    print(f"{region} Matrix Dimensions: {matrix.shape}")

Region 1 Matrix Dimensions: (120, 120)
Region 2 Matrix Dimensions: (120, 120)
Region 3 Matrix Dimensions: (120, 120)
Region 4 Matrix Dimensions: (120, 120)
Region 5 Matrix Dimensions: (120, 120)
Region 6 Matrix Dimensions: (120, 120)
Region 7 Matrix Dimensions: (120, 120)
Region 8 Matrix Dimensions: (120, 120)
Region 9 Matrix Dimensions: (120, 120)
Region 10 Matrix Dimensions: (120, 120)
Region 11 Matrix Dimensions: (120, 120)
Region 12 Matrix Dimensions: (120, 120)
Region 13 Matrix Dimensions: (120, 120)
Region 14 Matrix Dimensions: (120, 120)
Region 15 Matrix Dimensions: (120, 120)
Region 16 Matrix Dimensions: (120, 120)
Region 17 Matrix Dimensions: (120, 120)
Region 18 Matrix Dimensions: (120, 120)
Region 19 Matrix Dimensions: (120, 120)
Region 20 Matrix Dimensions: (120, 120)
Region 21 Matrix Dimensions: (120, 120)
Region 22 Matrix Dimensions: (120, 120)
Region 23 Matrix Dimensions: (120, 120)
Region 24 Matrix Dimensions: (120, 120)
Region 25 Matrix Dimensions: (120, 120)
Region 26

In [ ]:
for i in range(num_regions):
    start_row = i * rows_per_region + use_rows_per_region
    end_row = start_row + use_rows_per_region
    start_col = i * cols_per_region
    end_col = start_col + use_cols_per_region

    print(f"Region {i+1}: Start Row: {start_row}, End Row: {end_row}")
    print(f"Region {i+1}: Start Col: {start_col}, End Col: {end_col}")

    if end_row > total_rows:
        end_row = total_rows
    if end_col > total_cols:
        end_col = total_cols

    use_table = T_csr_2022[start_row:end_row, start_col:end_col].tocoo().tocsr()
    print(f"Region {i+1} Use Table Non-Zero Elements: {use_table.nnz}")  # Print number of non-zero elements


Region 1: Start Row: 120, End Row: 240
Region 1: Start Col: 0, End Col: 120
Region 1 Use Table Non-Zero Elements: 14395
Region 2: Start Row: 360, End Row: 480
Region 2: Start Col: 240, End Col: 360
Region 2 Use Table Non-Zero Elements: 14395
Region 3: Start Row: 600, End Row: 720
Region 3: Start Col: 480, End Col: 600
Region 3 Use Table Non-Zero Elements: 14398
Region 4: Start Row: 840, End Row: 960
Region 4: Start Col: 720, End Col: 840
Region 4 Use Table Non-Zero Elements: 14399
Region 5: Start Row: 1080, End Row: 1200
Region 5: Start Col: 960, End Col: 1080
Region 5 Use Table Non-Zero Elements: 14398
Region 6: Start Row: 1320, End Row: 1440
Region 6: Start Col: 1200, End Col: 1320
Region 6 Use Table Non-Zero Elements: 14392
Region 7: Start Row: 1560, End Row: 1680
Region 7: Start Col: 1440, End Col: 1560
Region 7 Use Table Non-Zero Elements: 14392
Region 8: Start Row: 1800, End Row: 1920
Region 8: Start Col: 1680, End Col: 1800
Region 8 Use Table Non-Zero Elements: 14396
Region 9: S

In [11]:
# # This code calculates X but the output are negative values which is not correct
# from scipy.sparse import identity, csr_matrix
# from scipy.sparse.linalg import spsolve
# import pickle

# # Load the new sliced T and Y matrices
# with open('Tsliced_2022.pkl', 'rb') as f:
#     Tsliced_2022 = pickle.load(f)
# with open('Ysliced_2022.pkl', 'rb') as f:
#     Ysliced_2022 = pickle.load(f)

# # Ensure the identity matrix matches the dimensions of the new T matrix
# I = identity(Tsliced_2022.shape[0], format='csr')  # Identity matrix of the same size as the new T

# # Formulating (I - T) for the new sliced T matrix
# A = I - Tsliced_2022

# # Solve (I - T)X = Y directly for X using the new sliced T matrix
# X0 = spsolve(A, Ysliced_2022)

# # Save the computed X for later use
# with open('X0.pkl', 'wb') as f:
#     pickle.dump(X0, f)

# # Optionally, save the matrix A if you need it for further analysis or verification
# with open('A.pkl', 'wb') as f:
#     pickle.dump(A, f)

# # Print outputs for verification
# print("Total Output Vector X:", X0)
# print("Dimensions of X:", X0.shape)
# print("Technical Coefficients Matrix A saved.")

/Users/cmor7802/anaconda3/envs/framework/lib/python3.11/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:412: SparseEfficiencyWarning: splu converted its input to CSC format
  warn('splu converted its input to CSC format', SparseEfficiencyWarning)


In [ ]:
# import numpy as np
# import pickle

# # Load the dense T matrix and the sparse Y matrix
# with open('Tdsliced_2022.pkl', 'rb') as f:
#     T_dense = pickle.load(f)
# with open('Ysliced_2022.pkl', 'rb') as f:
#     Y_sparse = pickle.load(f)

# # Convert the sparse Y matrix to a dense format
# Y_dense = Y_sparse.todense()

# # Sum across columns of Y if it has multiple columns, ensuring it reduces to a vector
# if Y_dense.shape[1] > 1:
#     Y_vector = np.sum(Y_dense, axis=1)  # Sum across columns, should be a (n, 1) matrix
#     Y_vector = np.array(Y_vector).flatten()  # Flatten to make sure it's a 1D array
# else:
#     Y_vector = np.array(Y_dense).flatten()  # Just flatten if it's already effectively a vector

# # Check dimensions of Y_vector to ensure it matches the number of rows in T_dense
# if Y_vector.shape[0] != T_dense.shape[0]:
#     raise ValueError(f"Dimension mismatch: Y_vector has {Y_vector.shape[0]} elements; expected {T_dense.shape[0]}.")

# # Create an identity matrix of the same size as T
# I = np.eye(T_dense.shape[0])

# # Formulating (I - T) for the new sliced T matrix
# A = I - T_dense

# # Solve (I - T)X = Y directly for X using numpy's solve function for dense matrices
# X = np.linalg.solve(A, Y_vector)

# # Save the computed X for later use
# with open('X0_d2.pkl', 'wb') as f:
#     pickle.dump(X, f)

# # Print outputs for verification
# print("Total Output Vector X:", X)
# print("Dimensions of X:", X.shape)

Total Output Vector X: [-3.34790361e+11  1.99988481e+12  3.06660791e+11 ... -2.97436927e+11
  2.17929153e+12  5.75994241e+09]
Dimensions of X: (19680,)


In [ ]:
# print("Statistics of X:")
# print("Min:", np.min(X))
# print("Max:", np.max(X))
# print("Mean:", np.mean(X))

Statistics of X:
Min: -4101838634521083.5
Max: 4976056884082517.0
Mean: -162468675697.08078


In [ ]:
# Check the diagonal elements of A
diagonal_elements = np.diag(A)
if np.any(diagonal_elements <= 0):
    print("Warning: Non-positive diagonal elements found in matrix A.")

# Calculate the condition number to assess stability
condition_number = np.linalg.cond(A)
print("Condition number of matrix A:", condition_number)

# Optionally, look at the minimum diagonal element
print("Minimum diagonal element in A:", np.min(diagonal_elements))


## Example - loading .mat files of IO data

In [ ]:
import scipy.io
X0_2020 = scipy.io.loadmat('/Users/cmor7802/Documents/MATLAB/climate_risk/Results/Augmentation/180923/X0.mat')

In [ ]:
import numpy as np

print("Statistics of X:")
print("Min:", np.min(X0_2020))
print("Max:", np.max(X0_2020))

In [ ]:
import scipy.io
import scipy.sparse as sp
import pickle

# Load the MATLAB file
data = scipy.io.loadmat('/Users/cmor7802/Documents/MATLAB/climate_risk/Results/Augmentation/180923/X0.mat')

# Access the data; adjust 'X0' based on how it's named in the MATLAB workspace
X0_2020 = data['X0'] 

# Convert to a sparse matrix if it's not already and if it makes sense to do so
X0_2020_sparse = sp.csr_matrix(X0_2020)

# Save the sparse matrix using pickle
with open('X0_2020_sparse.pkl', 'wb') as f:
    pickle.dump(X0_2020_sparse, f)

# Optionally, to verify what's saved, load it back
with open('X0_2020_sparse.pkl', 'rb') as f:
    loaded_X0_2020_sparse = pickle.load(f)

# Print out some details to verify
print("Shape of the loaded sparse matrix:", loaded_X0_2020_sparse.shape)
print("Non-zero elements of the loaded sparse matrix:", loaded_X0_2020_sparse.nnz)


Shape of the loaded sparse matrix: (1801, 1)
Non-zero elements of the loaded sparse matrix: 1801
